In [ ]:
import os
import json
from dotenv import load_dotenv
from typing import TypedDict, List, Annotated, Dict, Literal
from datetime import datetime
from pydantic import BaseModel, Field
from operator import add

from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

from logger import Logger

from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
class TimeResult(BaseModel):
    """Схема ответа от Time агента"""
    schedule: dict[str, str] = Field(
        description="Расписание в формате {'год:месяц:число:час:минута': 'событие'}"
    )

class ValidationResult(BaseModel):
    """Схема ответа от Validation агента"""
    next: Literal['Time', 'END'] = Field(
        description="Решение: Time - доработать, END - все в норме"
    )
    reason: str = Field(
        description="Краткое обоснование решения"
    )
    
print()

In [ ]:
#--------------------------MODELS CREATION------------------------------
load_dotenv('api_keys.env')

GITHUB_PAT = os.getenv('GITHUB_PAT')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

GITHUB_URL = os.getenv('GITHUB_URL')

GPT_MODEL = os.getenv('GPT_MODEL')
GEMINI_MODEL = os.getenv('GEMINI_MODEL')

main_model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY,
    temperature=0.1,
    max_retries=2
)

sub_model = ChatOpenAI(
    model=GPT_MODEL,
    api_key=GITHUB_PAT,
    base_url=GITHUB_URL,
    max_retries=5
)

#------------------------------------------GRAPH LOGIC---------------------------------
class GraphState(TypedDict):
    schedule: dict[str, str]
    
    user_task: str
    val_query: str
    next_agent: str
    image: str
    
    content: str
    attempts: int
    
Logger.init_log_file('logs.txt')

def open_prompt(file_path: str = 'prompts.json') -> dict:
    with open(file_path, 'r') as f:
        return json.load(f)
    
PROMPTS = open_prompt()
    
def Time(state: GraphState) -> GraphState:
    Logger.write_log('Time manager', 'вход')
    
    prompt = json.dumps(PROMPTS['Time'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    val_query = state.get('val_query', '')
    
    structured_model = main_model.with_structured_output(TimeResult)
    response = structured_model.invoke([HumanMessage([prompt, val_query, user_query])])
    
    schedule = response.schedule
        
    Logger.write_log('Time manager', 'Создано расписание', schedule)
    
    return {
        'schedule': schedule
    }
    
def IMG(state: GraphState) -> GraphState:
    Logger.write_log('Image analyzer', 'вход')
    
    prompt = json.dumps(PROMPTS['IMG'], ensure_ascii=False, indent=2)
    user_query = state['image']
    message = HumanMessage(content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{user_query}"}
            }
        ])
    
    response = main_model.invoke([message])
    
    try:
        res = response.content[0]['text']
        
    except:
        res = response.content
        Logger.write_log('Image analyzer', 'Внимание!! Возможно, вы используете не предпологаемую кодом модель!!', 'Предпологается gemini-3.1-flash-lite')
    
    Logger.write_log('Image analyzer', 'Изображение проанализировано', res)
    
    return {
        'user_task': state['user_task'] + res
    }
    
def VAL(state: GraphState) -> GraphState:
    Logger.write_log('Validation agent', 'вход')
    
    prompt = json.dumps(PROMPTS['VAL'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    schedule = state['schedule']
    structured_model = sub_model.with_structured_output(ValidationResult)
    
    response = structured_model.invoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Расписание: {schedule}\nЗапрос пользователя: {user_query}"}
    ])
    
    next = response.next
    task = '' if next == 'END' else response.reason
    
    Logger.write_log('Validation agent', f'Передаю {next}', task)
    
    return {
        'val_query': task,
        'next_agent': next,
        'attempts': state['attempts'] + 1
    } 
    
def Visualizer(state: GraphState) -> GraphState:
    Logger.write_log('Visualizer', 'вход')
    
    prompt = json.dumps(PROMPTS['Visualizer'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    schedule = state['schedule']
    response = sub_model.invoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Расписание: {schedule}\nЗапрос: {user_query}"}
    ])
    
    try:
        res = response.content[0]['text'].split()
        Logger.write_log('Visualizer', 'Внимание!! Возможно, вы используете не предпологаемую кодом модель!!', 'Предпологается gpt-4.1-mini')
        
    except:
        res = response.content.split()
    
    Logger.write_log('Visualizer', res)
    
    return {
        'content': res
    }
    
def ENTER_router(state: GraphState) -> str:
    img = state.get('image', None)
    if img:
        return 'IMG'
    return 'Time'
    
def VAL_router(state: GraphState) -> str:
    if state['attempts'] < 5: return state['next_agent']
    return 'END'

memo = InMemorySaver()
workflow = StateGraph(GraphState)

workflow.add_node('IMG', IMG)
workflow.add_node('Time', Time)
workflow.add_node('VAL', VAL)
workflow.add_node('Visualizer', Visualizer)

workflow.set_conditional_entry_point(
    ENTER_router,
    {'IMG': 'IMG', 'Time': 'Time'}
)

workflow.add_edge('IMG', 'Time')
workflow.add_edge('Time', 'VAL')

workflow.add_conditional_edges(
    'VAL',
    VAL_router,
    {'Time': 'Time', 'END': 'Visualizer'}
)

workflow.set_conditional_entry_point
workflow.set_finish_point('Visualizer')

app = workflow.compile(checkpointer=memo)
config = {"configurable": {"thread_id": "session_1"}}


In [ ]:
import base64
with open('ex2.png', 'rb') as f:
    b64_str = base64.b64encode(f.read()).decode('utf-8')

res = app.invoke({'user_task': f'отжуманя в 10:00, приложил расписание', 'image': b64_str, 'attempts': 0}, config=config)